# 03 — Building the random catalogs

Random catalogs for the catalog of notebook 02.

- **Sky positions** are uniform inside the footprint of the lightcone. The footprint is known exactly: it is
  the union of the 60 sky patches the mock is made of.
- **Redshifts** are drawn, with replacement, from the redshifts of the data catalog.
- A contaminated catalog gives three data catalogs, each with its own randoms:

| `which` | data catalog | `id_outcome` |
|---|---|---|
| `"all"` | galaxies and interlopers | 1, 2, 3 |
| `"galaxies"` | galaxies with a correct redshift | 1, 2 |
| `"interlopers"` | galaxies with a wrong redshift | 3 |

Randoms N times as numerous as the data are written as N files, each of the size of its data catalog.

In [1]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")      # diffsky imports JAX; keep it off the GPU

import numpy as np
import matplotlib.pyplot as plt
from romanmock import plotting

plotting.use_style()

MOCK = "sdss_feniks_hizels_260710_09_07_2026"                          # version of the diffsky mock
ROOT = f"/global/cfs/cdirs/m4943/Simulations/RomanMocks/{MOCK}"        # products of romanmock for this mock
CATALOG = f"{ROOT}/catalogs/Every_z1p5_1p9.hdf5"                       # the data catalog (notebook 02)
RANDOMS = f"{ROOT}/randoms/Every_z1p5_1p9.hdf5"                        # -> Every_z1p5_1p9_<which>_<index>.hdf5

In [2]:
from romanmock import Catalog

cat = Catalog.read(CATALOG)
print(cat)
print(f"sky area: {cat.area_deg2:.2f} deg2 ({cat.info['area_source']})")
for which in ("all", "galaxies", "interlopers"):
    print(f"{which:<12s} {len(cat.subset(which)):>11,}")

<Catalog: 7,931,502 galaxies, 10 columns ['gal_id', 'ra', 'dec', 'redshift_true', 'redshift_rsd', 'redshift_assigned', 'id_outcome', 'n_det', 'line_assigned', 'line_bitmask']; F > 1.50e-16 (other lines > 8.00e-17); redshift_assigned in (1.5, 1.9)>
sky area: 2062.65 deg2 (footprint)


all            7,931,502


galaxies       7,410,700
interlopers      520,802


## The randoms

In [3]:
N_TIMES = 10

files = cat.make_random(
    which=["all", "galaxies", "interlopers"],
    n_times=N_TIMES,                                    # number of files per kind
    seed=2026,
    write_to=RANDOMS,
    verbose=False,
)
for which, paths in files.items():
    size = sum(os.path.getsize(p) for p in paths) / 1e9
    print(f"{which:<12s} {len(paths)} files, {size:6.2f} GB   {paths[0].name} ... {paths[-1].name}")

all          10 files,   1.90 GB   Every_z1p5_1p9_all_00.hdf5 ... Every_z1p5_1p9_all_09.hdf5
galaxies     10 files,   1.78 GB   Every_z1p5_1p9_galaxies_00.hdf5 ... Every_z1p5_1p9_galaxies_09.hdf5
interlopers  10 files,   0.13 GB   Every_z1p5_1p9_interlopers_00.hdf5 ... Every_z1p5_1p9_interlopers_09.hdf5


Random catalog `k` of a kind depends on `(seed, which, k)` only. More can be added later without remaking
the existing ones: `cat.make_random(which="all", n_times=10, first=10, seed=2026, write_to=...)`.

## Checks

Every random point is inside the footprint, and each set of randoms has the redshift distribution of its data
catalog.

In [4]:
from astropy.table import Table
from romanmock import BaseCatalog

footprint = BaseCatalog(cat.settings["base"]).footprint
z_column = cat.settings["z_column"]
z_edges = np.linspace(*cat.settings["z_range"], 9)

rows = []
for which, paths in files.items():
    data, ran = cat.subset(which), Catalog.read(paths[0])
    n_data, n_ran = (np.histogram(c[z_column], bins=z_edges)[0] for c in (data, ran))
    ratio = n_data[n_ran > 0] / n_ran[n_ran > 0]
    rows.append({"which": which, "data": len(data), "randoms per file": len(ran),
                 "randoms inside the footprint": f"{footprint.contains(ran['ra'], ran['dec']).mean():.4f}",
                 "data inside the footprint": f"{footprint.contains(data['ra'], data['dec']).mean():.4f}",
                 "n(z) data / randoms": f"{ratio.min():.3f} - {ratio.max():.3f}"})
print(footprint)
Table(rows)

<PatchFootprint: 60 patches, 2062.65 deg2, 57.85 < dec < 75.16>


which,data,randoms per file,randoms inside the footprint,data inside the footprint,n(z) data / randoms
str11,int64,int64,str6,str6,str13
all,7931502,7931502,1.0000,1.0000,0.999 - 1.001
galaxies,7410700,7410700,1.0000,1.0000,0.998 - 1.002
interlopers,520802,520802,1.0000,0.9999,0.991 - 1.008


A few data galaxies lie just outside the footprint: satellites of haloes at its edge. They can be removed with
`cat[footprint.contains(cat["ra"], cat["dec"])]` before the randoms are made.

## Using them

```python
data = Catalog.read(f"{ROOT}/catalogs/Every_z1p5_1p9.hdf5").subset("galaxies")
randoms = [Catalog.read(f"{ROOT}/randoms/Every_z1p5_1p9_galaxies_{k:02d}.hdf5") for k in range(10)]
data["ra"], data["dec"], data["redshift_assigned"]              # numpy arrays
randoms[0]["ra"], randoms[0]["dec"], randoms[0]["redshift_assigned"]
```

The files are plain HDF5 with the datasets `ra`, `dec` and `redshift_assigned`. From the command line:

```bash
romanmock random $ROOT/catalogs/Every_z1p5_1p9.hdf5 -n 10 --which all galaxies interlopers \
          --seed 2026 -o $ROOT/randoms/Every_z1p5_1p9.hdf5
```

## Notes

- The randoms follow the footprint of the lightcone, a region of the sky. The sky area `area_deg2` of notebook 02
  is a number and does not enter: for a mock with another footprint, pass `footprint=` to `make_random`.
- A cut on `ra` or `dec` made with `cuts=` in `make_catalog` is applied to the randoms as well.
- Redshifts drawn from the data carry the radial structure of the data into the randoms, which slightly
  suppresses the clustering of the modes along the line of sight.
- The randoms of `"all"` are not the sum of the other two: each set is drawn from the redshifts of its own
  data catalog.